# 假新闻分类实验

本 notebook 使用机器学习方法对假新闻进行分类

In [1]:
# 导入必要的库
import pandas as pd
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. 读取数据

In [2]:
# 从CSV文件读取数据
df = pd.read_csv('fakenews_data.csv')

# 查看数据基本信息
print("数据集形状:", df.shape)
print("\n数据集前5行:")
print(df.head())

print("\n标签分布:")
print(df['label'].value_counts())

数据集形状: (3000, 3)

数据集前5行:
   Unnamed: 0                                            content label
0        4265  #回龙观关注# 赶紧检查一下你家里的牙膏，如果是黑色条赶紧扔掉！买牙膏时注意牙膏管反面底部的...    谣言
1        8363  【权威访谈·乘势而上奋勇前进——王凯：抓重点补短板增强发展动能】8月30日起，人民日报、新华...    事实
2          34                               据说这是今天中午新浪微博上不去的原因……    谣言
3        7290  【星战角色乱入白宫护卫发言人】当地时间12月18日，美国华盛顿，白宫发言人欧内斯特举行新闻发...    事实
4        7814  你常记不住重要的事，倒是很容易把别人说的话放心上，要知道，你的所有努力，不过是为了让自己更好...    事实

标签分布:
label
事实    1533
谣言    1467
Name: count, dtype: int64


## 2. 数据预处理

In [3]:
# 提取文本和标签
texts = df['content'].fillna('')  # 处理可能的空值
labels_raw = df['label']

# 将标签转换为数值: 谣言 -> 1, 事实 -> 0
labels = labels_raw.map({'谣言': 1, '事实': 0})

print("标签转换结果:")
print(labels.value_counts())

标签转换结果:
label
0    1533
1    1467
Name: count, dtype: int64


## 3. 使用jieba分词

In [4]:
# 定义分词函数
def tokenize_text(text):
    """使用jieba对文本进行分词"""
    return ' '.join(jieba.cut(text))

# 对所有文本进行分词
print("正在进行分词处理...")
texts_tokenized = texts.apply(tokenize_text)

# 查看分词结果示例
print("\n分词结果示例:")
for i in range(3):
    print(f"原文: {texts[i][:50]}...")
    print(f"分词: {texts_tokenized[i][:50]}...")
    print()

Building prefix dict from the default dictionary ...


正在进行分词处理...


Dumping model to file cache C:\Users\Lenovo\AppData\Local\Temp\jieba.cache
Loading model cost 0.772 seconds.
Prefix dict has been built successfully.



分词结果示例:
原文: #回龙观关注# 赶紧检查一下你家里的牙膏，如果是黑色条赶紧扔掉！买牙膏时注意牙膏管反面底部的颜色条。...
分词: # 回龙观 关注 #   赶紧 检查一下 你 家里 的 牙膏 ， 如果 是 黑色 条 赶紧 扔掉 ！...

原文: 【权威访谈·乘势而上奋勇前进——王凯：抓重点补短板增强发展动能】8月30日起，人民日报、新华社、中央...
分词: 【 权威 访谈 · 乘势而上 奋勇前进 — — 王凯 ： 抓 重点 补短 板 增强 发展 动能 】 ...

原文: 据说这是今天中午新浪微博上不去的原因……...
分词: 据说 这是 今天 中午 新浪 微博上 不 去 的 原因 … …...



## 4. TF-IDF特征提取

In [5]:
# 使用TfidfVectorizer将文本转换为TF-IDF向量
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # 最多保留5000个特征词
    min_df=2,          # 最小文档频率
    max_df=0.95       # 最大文档频率
)

# 转换文本为TF-IDF矩阵
X = tfidf_vectorizer.fit_transform(texts_tokenized)
y = labels

print(f"TF-IDF矩阵形状: {X.shape}")
print(f"特征词数量: {len(tfidf_vectorizer.get_feature_names_out())}")

TF-IDF矩阵形状: (3000, 5000)
特征词数量: 5000


## 5. 划分训练集和测试集

In [6]:
# 按7:3比例划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42,
    stratify=y  # 保持标签分布一致
)

print(f"训练集大小: {X_train.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")
print(f"\n训练集标签分布:\n{y_train.value_counts()}")
print(f"\n测试集标签分布:\n{y_test.value_counts()}")

训练集大小: 2100
测试集大小: 900

训练集标签分布:
label
0    1073
1    1027
Name: count, dtype: int64

测试集标签分布:
label
0    460
1    440
Name: count, dtype: int64


## 6. 训练分类模型

In [7]:
### 6.1 Logistic Regression 分类器

In [8]:
# 训练Logistic Regression模型
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# 在测试集上预测
y_pred_lr = lr_model.predict(X_test)
# 输出分类报告
print("=" * 50)
print("Logistic Regression 分类报告")
print("=" * 50)
print(classification_report(y_test, y_pred_lr, target_names=['事实', '谣言']))

Logistic Regression 分类报告
              precision    recall  f1-score   support

          事实       0.81      0.84      0.82       460
          谣言       0.82      0.79      0.81       440

    accuracy                           0.81       900
   macro avg       0.82      0.81      0.81       900
weighted avg       0.81      0.81      0.81       900



In [ ]:
### 6.2 Naive Bayes 分类器

In [9]:
# 训练Naive Bayes模型
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 在测试集上预测
y_pred_nb = nb_model.predict(X_test)
# 输出分类报告
print("=" * 50)
print("Naive Bayes 分类报告")
print("=" * 50)
print(classification_report(y_test, y_pred_nb, target_names=['事实', '谣言']))

Naive Bayes 分类报告
              precision    recall  f1-score   support

          事实       0.81      0.83      0.82       460
          谣言       0.82      0.80      0.81       440

    accuracy                           0.82       900
   macro avg       0.82      0.82      0.82       900
weighted avg       0.82      0.82      0.82       900



## 7. 模型对比总结

In [10]:
# 计算准确率对比
from sklearn.metrics import accuracy_score

lr_accuracy = accuracy_score(y_test, y_pred_lr)
nb_accuracy = accuracy_score(y_test, y_pred_nb)

print("=" * 50)
print("模型准确率对比")
print("=" * 50)
print(f"Logistic Regression 准确率: {lr_accuracy:.4f}")
print(f"Naive Bayes 准确率: {nb_accuracy:.4f}")

if lr_accuracy > nb_accuracy:
    print("\n结论: Logistic Regression 模型表现更好")
else:
    print("\n结论: Naive Bayes 模型表现更好")

模型准确率对比
Logistic Regression 准确率: 0.8144
Naive Bayes 准确率: 0.8156

结论: Naive Bayes 模型表现更好
